# Ballon d'Or: initial data exploration

**Question:** How accurately can player performance statistics predict nominee vote shares in unseen award years?

This notebook audits the source workbook, proposes the first feature set, and identifies preprocessing needs. It does not train a model, fit preprocessing, or modify the source data.

**Source:** [original workbook](../Data/raw/Ballon_dOr_2015_2025.xlsx), especially `TrainingData`, `InputDataset`, `OutputDataset`, `DataDictionary`, and `ReadMe`. Definitions and limitations below come from those sheets.

Run cells from top to bottom. Launch Jupyter from the repository root or this notebook's folder. For setup, see the [repository README](../README.md).


In [1]:
from pathlib import Path
import platform

import numpy as np
import openpyxl
import pandas as pd
from IPython.display import display

# Locate the project without depending on a machine-specific absolute path.
repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "Data/raw/Ballon_dOr_2015_2025.xlsx").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/.")

workbook_path = repo_root / "Data/raw/Ballon_dOr_2015_2025.xlsx"
sheets = pd.read_excel(workbook_path, sheet_name=None, engine="openpyxl")
data = sheets["TrainingData"].copy()
print(f"Python {platform.python_version()} | pandas {pd.__version__} | openpyxl {openpyxl.__version__}")
display(pd.DataFrame([
    {"sheet": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in sheets.items()
]))


Python 3.12.14 | pandas 2.2.3 | openpyxl 3.1.5


,sheet,rows,columns
0,TrainingData,293,16
1,InputDataset,293,40
2,OutputDataset,293,15
3,CompetitionData,1045,13
4,Coverage,10,8
5,DataDictionary,29,3
6,ReadMe,23,2


## 1. Define column roles

Start with six numeric performance features, position, and league. Player names and IDs are identifiers; year and season define cohorts. None of these identifiers is an initial predictor. `target_vote_share` is already normalized and must not be included in the inputs or normalized again.

Nationality is context rather than on-field performance. Champions League stage is a possible later team-achievement comparison; the winner flag duplicates the stage's Winner category. Rank, points, percentages, and provenance fields are excluded from predictors.


In [2]:
numeric_features = [
    "league_appearances", "league_minutes_played", "league_goals",
    "league_assists", "league_yellow_cards", "league_red_cards",
]
categorical_features = ["position", "league"]
identifier_columns = ["year", "player", "season", "player_id"]
target = "target_vote_share"
excluded_context = ["nationality", "ucl_team_stage", "ucl_team_winner"]
feature_columns = numeric_features + categorical_features
required_columns = identifier_columns + feature_columns + [target]

missing_columns = sorted(set(required_columns) - set(data.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert not set(feature_columns) & set(identifier_columns + [target])

role_rows = []
for role, columns in [
    ("Numeric input", numeric_features),
    ("Categorical input", categorical_features),
    ("Identifier / split only", identifier_columns),
    ("Target only", [target]),
    ("Excluded from initial model", excluded_context),
]:
    role_rows.extend({"column": column, "role": role, "dtype": str(data[column].dtype)}
                     for column in columns)
display(pd.DataFrame(role_rows))
display(data[required_columns].head())


,column,role,dtype
0,league_appearances,Numeric input,int64
1,league_minutes_played,Numeric input,int64
2,league_goals,Numeric input,int64
3,league_assists,Numeric input,int64
4,league_yellow_cards,Numeric input,int64
5,league_red_cards,Numeric input,int64
6,position,Categorical input,object
7,league,Categorical input,object
8,year,Identifier / split only,int64
9,player,Identifier / split only,object


,year,player,season,player_id,league_appearances,league_minutes_played,league_goals,league_assists,league_yellow_cards,league_red_cards,position,league,target_vote_share
0,2015,Lionel Messi,2014/15,28003,38,3375,43,21,4,0,Forward,La Liga,0.4133
1,2015,Cristiano Ronaldo,2014/15,8198,35,3099,48,17,4,1,Forward,La Liga,0.2776
2,2015,Neymar,2014/15,68290,33,2571,22,9,6,0,Forward,La Liga,0.0786
3,2015,Robert Lewandowski,2014/15,38253,31,2487,17,7,0,0,Forward,Bundesliga,0.0417
4,2015,Luis Suárez,2014/15,44352,27,2179,16,16,4,0,Forward,La Liga,0.0338


## 2. Audit completeness, keys, and numeric validity

These are integrity checks across the workbook, including reserved years. They do not fit transformations or compare candidate features with held-out outcomes. Repeated players across different years are expected; repeated player-year keys are not.


In [3]:
numeric_columns = ["year", "player_id", *numeric_features, "ucl_team_winner", target]
checks = {
    "Required values are present": bool(data[required_columns].notna().all().all()),
    "No duplicate rows": not bool(data.duplicated().any()),
    "Unique year + player ID": not bool(data.duplicated(["year", "player_id"]).any()),
    "Unique year + player name": not bool(data.duplicated(["year", "player"]).any()),
    "Numeric columns contain numeric types": all(
        pd.api.types.is_numeric_dtype(data[column]) for column in numeric_columns
    ),
}
if checks["Numeric columns contain numeric types"]:
    checks.update({
        "Numeric values are finite": bool(np.isfinite(data[numeric_columns].to_numpy()).all()),
        "Performance counts are nonnegative": bool(data[numeric_features].ge(0).all().all()),
        "Performance counts are whole numbers": bool(data[numeric_features].mod(1).eq(0).all().all()),
        "Vote shares are between 0 and 1": bool(data[target].between(0, 1).all()),
        "Minutes and appearances are positive": bool(
            data[["league_minutes_played", "league_appearances"]].gt(0).all().all()
        ),
        "UCL winner flag is binary": bool(data["ucl_team_winner"].isin([0, 1]).all()),
    })

display(pd.DataFrame([{"check": name, "passed": passed} for name, passed in checks.items()]))
display(data.isna().sum().rename("missing_count").to_frame())
assert all(checks.values()), "A data integrity check failed; investigate before modelling."


,check,passed
0,Required values are present,True
1,No duplicate rows,True
2,Unique year + player ID,True
3,Unique year + player name,True
4,Numeric columns contain numeric types,True
5,Numeric values are finite,True
6,Performance counts are nonnegative,True
7,Performance counts are whole numbers,True
8,Vote shares are between 0 and 1,True
9,Minutes and appearances are positive,True


,missing_count
year,0
player,0
season,0
player_id,0
position,0
league,0
nationality,0
league_appearances,0
league_minutes_played,0
league_goals,0


In [4]:
# Confirm that the core inputs and labels agree with their supporting sheets.
input_columns = identifier_columns + feature_columns + excluded_context
pd.testing.assert_frame_equal(
    data[input_columns].sort_values(["year", "player_id"]).reset_index(drop=True),
    sheets["InputDataset"][input_columns].sort_values(["year", "player_id"]).reset_index(drop=True),
)
label_check = data[["year", "player", target]].merge(
    sheets["OutputDataset"][["year", "player", target]],
    on=["year", "player"], how="outer", suffixes=("_training", "_output"),
    validate="one_to_one", indicator=True,
)
assert label_check["_merge"].eq("both").all(), "Unmatched player-year labels."
assert np.allclose(label_check[f"{target}_training"], label_check[f"{target}_output"])
print("Core inputs agree with InputDataset; every label matches OutputDataset.")


Core inputs agree with InputDataset; every label matches OutputDataset.


## 3. Check year coverage and reserve later years

Split: train through 2023 and test on 2024-2025. The 2020 award is absent by design. This is a starting split, not a trained model. All fitted transformations will use training rows only.

Vote shares should sum approximately to 1 within a year. The workbook documents that 2015 uses published weighted percentages with a small rounding difference; later years use points divided by the year's total points. A tolerance of 0.0001 accommodates the documented 2015 difference.


In [5]:
cohorts = data.groupby("year").agg(
    nominees=("player_id", "size"),
    unique_players=("player_id", "nunique"),
    vote_share_sum=(target, "sum"),
)
cohorts["sum_near_one"] = np.isclose(cohorts["vote_share_sum"], 1.0, atol=1e-4, rtol=0)
display(cohorts)
assert cohorts["sum_near_one"].all(), "Unexpected vote-share total."

train = data.loc[data["year"] <= 2023].copy()
test = data.loc[data["year"] >= 2024].copy()
splits = {"train": train, "test": test}
display(pd.DataFrame([
    {"split": name, "rows": len(frame), "years": ", ".join(map(str, sorted(frame["year"].unique())))}
    for name, frame in splits.items()
]))
assert all(len(frame) for frame in splits.values()), "A proposed split is empty."
assert len(train) + len(test) == len(data)
assert train["year"].max() < test["year"].min()


,nominees,unique_players,vote_share_sum,sum_near_one
year,,,,
2015,23,23,1.000061,True
2016,30,30,1.000000,True
2017,30,30,1.000000,True
2018,30,30,1.000000,True
2019,30,30,1.000000,True
2021,30,30,1.000000,True
2022,30,30,1.000000,True
2023,30,30,1.000000,True
2024,30,30,1.000000,True


,split,rows,years
0,train,233,"2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023"
1,test,60,"2024, 2025"


## 4. Inspect training distributions and category labels

Use the training portion for exploratory summaries. Numeric features have different scales, and the first regression keeps them in their original units. Correlation among counts does not establish causal effects or make coefficients independent measures of importance.

Position will use one-hot encoding with a reference category for linear regression. League cells may contain multiple leagues: split on `;`, trim spaces, and create one indicator per league. For example, `Premier League; Bundesliga` belongs to both leagues. Do not treat each combination as a new league.


In [6]:
display(train[numeric_features + [target]].describe().T)
display(train["position"].value_counts().rename_axis("position").rename("training_rows").to_frame())
display(train["league"].value_counts().rename_axis("league_label").rename("training_rows").to_frame())

for column in categorical_features:
    assert data[column].map(lambda value: isinstance(value, str) and bool(value.strip())).all()
    assert data[column].eq(data[column].str.strip()).all(), f"Whitespace in {column}."

multi_league_rows = data.loc[data["league"].str.contains(";", regex=False), ["year", "player", "league"]]
print(f"Multi-league records requiring split indicators: {len(multi_league_rows)}")
display(multi_league_rows)

training_leagues = train["league"].str.split(";").explode().str.strip()
display(training_leagues.value_counts().rename_axis("league").rename("training_memberships").to_frame())
print("Membership counts can exceed row counts because a player can represent multiple leagues.")


,count,mean,std,min,25%,50%,75%,max
league_appearances,233.0,31.909871,4.229977,17.0,29.000000,32.000000,35.000000,38.000000
league_minutes_played,233.0,2582.527897,478.235857,974.0,2247.000000,2621.000000,2970.000000,3420.000000
league_goals,233.0,13.291845,11.228073,0.0,3.000000,12.000000,22.000000,48.000000
league_assists,233.0,6.733906,4.932957,0.0,2.000000,7.000000,10.000000,21.000000
league_yellow_cards,233.0,3.442060,2.477022,0.0,2.000000,3.000000,5.000000,11.000000
league_red_cards,233.0,0.068670,0.253436,0.0,0.000000,0.000000,0.000000,1.000000
target_vote_share,233.0,0.034335,0.075664,0.0,0.000735,0.004616,0.017045,0.478484


,training_rows
position,
Forward,125
Midfielder,58
Defender,29
Goalkeeper,21


,training_rows
league_label,
Premier League,79
La Liga,71
Serie A,32
Bundesliga,24
Ligue 1,15
Eredivisie,5
Primeira Liga,3
Premier League; Bundesliga,1
Premier League; Primeira Liga,1


Multi-league records requiring split indicators: 6


,year,player,league
46,2016,Kevin De Bruyne,Premier League; Bundesliga
170,2021,Rúben Dias,Premier League; Primeira Liga
191,2022,Luis Díaz,Primeira Liga; Premier League
192,2022,Cristiano Ronaldo,Premier League; Serie A
274,2025,Khvicha Kvaratskhelia,Serie A; Ligue 1
280,2025,Scott McTominay,Serie A; Premier League


,training_memberships
league,
Premier League,83
La Liga,71
Serie A,33
Bundesliga,25
Ligue 1,15
Primeira Liga,5
Eredivisie,5


Membership counts can exceed row counts because a player can represent multiple leagues.


In [7]:
# Inspect skew using training outcomes only; do not tune on final test outcomes.
vote_bins = pd.cut(
    train[target], bins=[-np.inf, 0, 0.01, 0.05, 0.10, 1.0],
    labels=["Zero", "Above 0 to 1%", "Above 1% to 5%", "Above 5% to 10%", "Above 10%"],
)
display(vote_bins.value_counts(sort=False).rename("training_rows").to_frame())
print("Many nominees receive small shares: report ranking quality alongside MAE, RMSE, and R-squared.")


,training_rows
target_vote_share,
Zero,36
Above 0 to 1%,116
Above 1% to 5%,42
Above 5% to 10%,13
Above 10%,26


Many nominees receive small shares: report ranking quality alongside MAE, RMSE, and R-squared.


## 5. Audit optional feature coverage

Optional advanced features are not part of the first model. The following counts identify collection gaps, not values to replace with zero. Source URLs, coverage flags, and incomplete `covered_club_*` totals are not initial performance predictors.

When defensive stats are added, document the metric, competition scope, units, source, and season coverage. Join extra features using year + player ID and validate one match per player-year. Goalkeeper-specific metrics may require separate treatment rather than assuming an unavailable outfield statistic equals zero.


In [8]:
optional_features = [
    "league_xG", "league_xA", "league_shots", "league_key_passes",
    "league_xGChain", "league_xGBuildup", "ucl_appearances",
    "ucl_minutes_played", "ucl_goals", "ucl_assists", "ucl_knockout_goals",
]
optional_data = sheets["InputDataset"]
coverage = pd.DataFrame({
    "available_rows": optional_data[optional_features].notna().sum(),
    "missing_rows": optional_data[optional_features].isna().sum(),
    "missing_percent": optional_data[optional_features].isna().mean().mul(100).round(2),
})
display(coverage)
display(optional_data[optional_features].isna().groupby(optional_data["year"]).sum())

dictionary = sheets["DataDictionary"]
display(dictionary.loc[dictionary["Role"].eq("Deferred"), ["Field", "Definition"]])


,available_rows,missing_rows,missing_percent
league_xG,282,11,3.75
league_xA,282,11,3.75
league_shots,282,11,3.75
league_key_passes,282,11,3.75
league_xGChain,282,11,3.75
league_xGBuildup,282,11,3.75
ucl_appearances,288,5,1.71
ucl_minutes_played,288,5,1.71
ucl_goals,288,5,1.71
ucl_assists,288,5,1.71


,league_xG,league_xA,league_shots,league_key_passes,league_xGChain,league_xGBuildup,ucl_appearances,ucl_minutes_played,ucl_goals,ucl_assists,ucl_knockout_goals
year,,,,,,,,,,,
2015,0,0,0,0,0,0,2,2,2,2,2
2016,1,1,1,1,1,1,1,1,1,1,1
2017,0,0,0,0,0,0,0,0,0,0,0
2018,0,0,0,0,0,0,0,0,0,0,0
2019,5,5,5,5,5,5,0,0,0,0,0
2021,1,1,1,1,1,1,2,2,2,2,2
2022,3,3,3,3,3,3,0,0,0,0,0
2023,0,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0,0


,Field,Definition
24,Successful Tackles / Interceptions / Defensive...,No consistent all-competition source establish...
25,Team Trophies,Comprehensive trophy count not collected. Use ...
26,Big Game Goals,Requires an agreed competition/round definitio...
27,International Team Progression / national-team...,Historical national-team appearances absent fr...


## 6. Findings and next implementation step

- The initial dataset has 293 nominee-season rows across 10 award years, with no missing core values or duplicate player-year keys. The split contains 233 training and 60 test rows.
- Initial predictors: six domestic-league numeric features plus position and league. Keep `target_vote_share` unchanged as the target. Keep identifiers for joins, splitting, and reporting only.
- Six records contain multiple leagues and need separate league indicators. Category handling must also support leagues absent from the training years.
- Six advanced league features each have 11 missing rows; individual UCL features each have five missing rows. No defensive metrics or national-team performance are currently supplied.
- The 2015 vote-share total is approximately 1.000061, consistent with the workbook's documented precision difference. Do not silently renormalize it.
- This is a nominee-only, league-based sample. Earlier awards used a different performance window; international performances and some competitions are omitted. Vote-share normalization does not remove changes in voting rules. Prediction errors do not establish popularity effects or bias.
- **Model implementation:** `scripts/train.py` encodes the initial categories, fits linear regression on training years, and compares train/test predictions with a training-mean baseline. The core data needs no imputation, and numeric inputs remain in their original units.

No cleaned data, fitted preprocessing, model, or prediction files are generated by this notebook. The source workbook remains unchanged.

**AI assistance:** Codex assisted with creating this notebook and its initial data audit. The team should review the analysis and include its use in the course's required disclosure/reflection.
